# Exploración de corpus del español de Colombia con `open_corpus_co_es`

**Curso:** Procesamiento de Lenguaje Natural · Unidad 2 — Patrones estadísticos del lenguaje  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 2 — Words and Tokens  
**Librería:** [`open_corpus_co_es`](https://pypi.org/project/open_corpus_co_es/) 0.1.1a4 (L. G. Moreno Sandoval, Pontificia Universidad Javeriana)  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook usa la librería **`open_corpus_co_es`**, un catálogo de corpus en español de Colombia y la región (discursos, noticias, redes sociales, léxicos afectivos), para hacer **análisis exploratorio de texto**: buscar y descargar corpus de forma reproducible, medir su tamaño y diversidad léxica, comprobar las leyes de Zipf y Heaps, encontrar el vocabulario distintivo de cada país y calcular perfiles emocionales con un léxico. Es el paso de “conocer los datos” antes de construir cualquier modelo.

**Objetivo:** aprender a elegir y caracterizar un corpus con estadísticas sólidas, de modo que las decisiones posteriores (tamaño del vocabulario, limpieza, comparación entre fuentes) se basen en evidencia y no en suposiciones.

## Ejemplos de Aplicación en el Mundo Real

**1. Salud — Vigilancia de conversaciones sobre salud pública.** Una secretaría de salud compara el vocabulario de publicaciones en redes sobre salud en Colombia y otros países (corpus `salud_*_2024` del catálogo) para detectar temas emergentes, como brotes o desinformación sobre vacunas.

**2. Turismo y hospitalidad — Vocabulario distintivo por destino.** Un observatorio de turismo aplica el *log-odds* con prior de Dirichlet (sección 5.2) a reseñas de Cartagena, Santa Marta y San Andrés para descubrir qué palabras caracterizan a cada destino y orientar su promoción.

**3. Finanzas y servicio al cliente — Monitoreo de noticias económicas.** Un área de riesgo de un banco mide qué términos sobresalen en las noticias de cada país (como en la sección 5) para anticipar temas regulatorios o de mercado que afecten a sus clientes.

**4. Energía e industria — Perfil afectivo de la opinión sobre proyectos.** Una empresa de energía aplica un léxico afectivo (sección 6) a comunicados y noticias sobre sus proyectos para seguir la carga emocional del debate público, sabiendo que este método no capta ironía ni negación.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install -U numpy pandas matplotlib nltk
pip install --user -U "open_corpus_co_es==0.1.1a4"   # versión preliminar: la versión fija es obligatoria
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `re, collections.Counter, pathlib.Path` | Tokenización con expresiones regulares, conteos y rutas |
| `numpy, pandas` | Cálculo y tablas |
| `matplotlib.pyplot` | Gráficas de Zipf, Heaps y mapa de calor |
| `nltk` | Stopwords en español y concordancias (`Text.concordance`) |
| `open_corpus_co_es` | Catálogo, descarga con verificación (*checksum*) y carga de corpus |

**Datos:** se descargan de Google Drive la primera vez a `~/.open_corpus_co_es/data/<nombre>/`; las ejecuciones siguientes usan la copia local. Las noticias de la sección 5 ocupan ~40–50 MB por país.

## Marco metodológico: pipeline ML / LLM (4×3)

![Pipeline ML y LLM](img/Pipeline_ML_LLM_4x3.png)

El diagrama tiene una franja común de datos (arriba), dos rutas de modelado y una franja de operación (abajo): **ruta clásica** (*feature-based ML*) y **ruta LLM** (*foundation models*). Este notebook recorre la **ruta clásica**. Etapas del diagrama que cubre:

| Etapa del diagrama | Dónde se aborda en este notebook |
|--------------------|----------------------------------|
| Business & Data Understanding | Secciones 2–6: catálogo, estadísticas de corpus, leyes de Zipf y Heaps, vocabulario distintivo, perfil afectivo |
| Data | Secciones 2–3: selección en el catálogo y descarga reproducible con verificación |
| Data Preprocessing | Sección 3: función `tokenizar` (minúsculas, solo letras, con tildes y ñ) y eliminación de stopwords |

Es un notebook de **entendimiento de los datos**: no entrena modelos. El ejercicio 5 conecta con la ruta clásica de modelado (TF-IDF + regresión logística).

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **IEEE 7003-2024** — Consideraciones sobre sesgo algorítmico | Comparar vocabulario entre países y fuentes revela **diferencias de representación** en los datos antes de entrenar un modelo con ellos. |
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | Riesgos documentados: la TTR depende de la longitud (se corrige con una ventana fija) y los léxicos ignoran negación e ironía (sección 6). |
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | Todas las estadísticas se calculan con fórmulas explícitas (Zipf, Heaps, *log-odds*), sin cajas negras. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | Librería y corpus abiertos con descarga verificada por *checksum*: el análisis es reproducible por terceros. |

## Objetivos de aprendizaje

1. Explorar un **catálogo de corpus** y elegir conjuntos de datos por tipo, tema, país y tamaño.
2. Descargar y cargar corpus de forma reproducible (verificación por *checksum*, caché local).
3. Medir estadísticas básicas de un corpus: **tokens, tipos y razón tipo/token**.
4. Comprobar la **ley de Zipf** y la **ley de Heaps** en texto real en español.
5. Comparar el **vocabulario entre países** (noticias de Colombia, Perú y Ecuador).
6. Aplicar un **léxico afectivo** a discursos presidenciales.

> Los archivos se descargan de Google Drive a `~/.open_corpus_co_es/data/<nombre>/` la primera vez; las ejecuciones siguientes reutilizan la copia local.

---

## 1. Configuración del entorno

In [ ]:
# Biblioteca estándar
import re
from collections import Counter
from pathlib import Path

# Librerías de terceros
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.text import Text

from open_corpus_co_es import download_corpus, list_corpus, load_corpus
from open_corpus_co_es.downloader import CATALOG, get_corpus_path

# Recursos de NLTK (solo la primera vez) y stopwords en español
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
STOP_ES = set(stopwords.words('spanish'))

pd.set_option('display.max_colwidth', 80)
print("Listo.")

---

## 2. El catálogo

`list_corpus()` devuelve un diccionario `{nombre: metadatos}` solo con las entradas **habilitadas**.
Lo convertimos en un DataFrame para filtrarlo y ordenarlo.

In [ ]:
# Catálogo de corpus habilitados como tabla, con tamaño en MB
inventario = list_corpus()

catalogo = (
    pd.DataFrame.from_dict(inventario, orient='index')
      .assign(MB=lambda d: (d['bytes'].fillna(0) / 1e6).round(1),
              filas=lambda d: d['filas'].fillna(d['documentos']))
      [['tipo', 'origen', 'tema', 'pais', 'año', 'extension', 'filas', 'MB', 'descripcion']]
      .sort_values(['tipo', 'origen', 'tema'])
)
print(f"Entradas habilitadas: {len(catalogo)}")
catalogo.head(10)

In [ ]:
# ¿Cómo se distribuye el catálogo por tipo y origen?
resumen = (catalogo.groupby(['tipo', 'origen'])
                   .agg(entradas=('tema', 'size'), filas=('filas', 'sum'), MB=('MB', 'sum'))
                   .sort_values('MB', ascending=False))
resumen

In [ ]:
# Filtro de ejemplo: corpus colombianos pequeños (< 5 MB), útiles para experimentos rápidos en clase
catalogo.query("tipo == 'corpora' and pais == 'Colombia' and MB < 5")[['tema', 'filas', 'MB', 'descripcion']]

### 2.1 Corpus etiquetados para tareas supervisadas

Varias entradas tienen una **columna de etiqueta**, así que sirven para unidades posteriores (clasificación, modelos de secuencia, transformers):

| Corpus | Columna de texto | Etiqueta(s) | Filas | Tarea posible |
|---|---|---|---|---|
| `es_hs_project_raw_extended_v2` | `text` | `HS`, `TR`, `AG` | 4 950 | Discurso de odio (estilo HatEval) |
| `discurso_odio` | `text` | `HS`, `RS`, `MS` | 915 939 | Discurso de odio a gran escala |
| `llm_humano` | `Texto` | `Clasificacion` | 1 844 | Texto humano frente a texto generado por LLM |
| `edades_entrenamiento` | `text` | `age_range` | 169 336 | Perfilado de autor (edad) |
| `news_2020_*_v2` | `text` | `topic` | 17k–27k | Clasificación temática |
| `salud_/educacion_/seguridad_*_2024` | `Hit Sentence` | `Sentiment` | 20 000 cada uno | Sentimiento (redes sociales) |

---

## 3. Función auxiliar de carga

`load_corpus()` convierte cada fila en un `dict` (`{"text": ..., **metadatos}`). Es cómodo para corpus pequeños, pero lento para los grandes (por ejemplo `discurso_odio`, 916 000 filas). La función de abajo descarga con la librería (con verificación por *checksum*) y luego lee el archivo **directamente con pandas**.

In [ ]:
def cargar_df(nombre: str, force: bool = False) -> pd.DataFrame:
    """Descarga (si hace falta) una entrada del catálogo y la devuelve como DataFrame."""
    download_corpus(nombre, force=force)
    meta = CATALOG[nombre]
    ext = meta['extension'].lower()
    ruta = Path(get_corpus_path(nombre)) / f"{nombre}.{ext}"
    if ext == 'parquet':
        return pd.read_parquet(ruta)
    if ext == 'xlsx':
        return pd.read_excel(ruta)
    if ext == 'csv':
        return pd.read_csv(ruta, sep=None, engine='python')
    raise ValueError(f"Extensión no soportada: {ext}")


# Token = secuencia de letras (incluye tildes, ü y ñ); descarta números y puntuación
TOKEN_RE = re.compile(r"[a-záéíóúüñ]+", re.IGNORECASE)

def tokenizar(texto: str) -> list[str]:
    """Tokens alfabéticos en minúsculas (incluye tildes y ñ)."""
    return TOKEN_RE.findall(str(texto).lower())

---

## 4. Caso de estudio 1 — Discursos presidenciales de Colombia (`presidentes`)

Los mismos pasos de la guía rápida de la librería.

In [ ]:
# Descarga y carga el corpus como lista de diccionarios
nombre = "presidentes"
download_corpus(nombre)
documentos = load_corpus(nombre)

print(len(documentos))
print(documentos[0]["text"][:200])
print("Campos de metadatos:", [k for k in documentos[0] if k != 'text'])

In [ ]:
# Tokens, tipos y razón tipo/token (TTR) de cada discurso
pres = pd.DataFrame(documentos)
pres['tokens'] = pres['text'].map(tokenizar)
pres['n_tokens'] = pres['tokens'].map(len)
pres['n_types'] = pres['tokens'].map(lambda t: len(set(t)))
pres['TTR'] = (pres['n_types'] / pres['n_tokens']).round(3)

pres[['nombre', 'fecha_inicio', 'fecha_fin', 'n_tokens', 'n_types', 'TTR']].sort_values('fecha_inicio')

> **Pregunta:** la TTR baja a medida que los textos son más largos, así que comparar la TTR bruta entre discursos de distinta longitud es engañoso.
> Una comparación más justa usa una ventana fija (por ejemplo, los primeros 1 000 tokens), como en la siguiente celda.

In [ ]:
# TTR sobre los primeros 1 000 tokens; NaN si el discurso es más corto
VENTANA = 1000
pres['TTR_1000'] = pres['tokens'].map(lambda t: len(set(t[:VENTANA])) / VENTANA if len(t) >= VENTANA else np.nan)
pres[['nombre', 'n_tokens', 'TTR', 'TTR_1000']].sort_values('TTR_1000', ascending=False)

In [ ]:
# Palabras de contenido más frecuentes (sin stopwords ni palabras de 1–2 letras)
frec = Counter(w for toks in pres['tokens'] for w in toks if w not in STOP_ES and len(w) > 2)
pd.DataFrame(frec.most_common(25), columns=['palabra', 'frecuencia'])

In [ ]:
# Concordancias: cada aparición de "paz" con su contexto (keyword in context)
texto_nltk = Text(w for toks in pres['tokens'] for w in toks)
texto_nltk.concordance('paz', width=90, lines=12)

### 4.1 Ley de Zipf

Zipf: la frecuencia de una palabra es aproximadamente inversamente proporcional a su posición en el ranking, $f(r) \propto r^{-\alpha}$ con $\alpha \approx 1$.
En un gráfico log–log es una recta con pendiente $-\alpha$.

In [ ]:
def zipf(tokens, titulo):
    # Frecuencias ordenadas de mayor a menor y ajuste lineal en escala log–log
    freqs = np.array(sorted(Counter(tokens).values(), reverse=True))
    rangos = np.arange(1, len(freqs) + 1)
    alpha, c = np.polyfit(np.log(rangos), np.log(freqs), 1)
    plt.figure(figsize=(6, 4))
    plt.loglog(rangos, freqs, '.', ms=3, label='observado')
    plt.loglog(rangos, np.exp(c) * rangos ** alpha, '-', label=f'ajuste  pendiente = {alpha:.2f}')
    plt.xlabel('posición (rango)'); plt.ylabel('frecuencia'); plt.title(titulo); plt.legend(); plt.show()
    return alpha

todos = [w for toks in pres['tokens'] for w in toks]
zipf(todos, 'Zipf — discursos presidenciales');

---

## 5. Caso de estudio 2 — Noticias de Colombia, Perú y Ecuador (2020)

Tres corpus comparables (`_id`, `topic`, `text`) de ~40–50 MB cada uno. La descarga puede tardar un minuto.

In [ ]:
# Un corpus de noticias por país, unidos en una sola tabla
PAISES = {'Colombia': 'news_2020_colombia_v2', 'Perú': 'news_2020_peru_v2', 'Ecuador': 'news_2020_ecuador_v2'}

noticias = pd.concat(
    [cargar_df(n).assign(pais=p) for p, n in PAISES.items()],
    ignore_index=True,
)
# Tokenización y longitud de cada noticia
noticias['tokens'] = noticias['text'].map(tokenizar)
noticias['n_tokens'] = noticias['tokens'].map(len)

# Documentos, tokens totales y mediana de tokens por documento, por país
noticias.groupby('pais').agg(documentos=('text', 'size'),
                             tokens=('n_tokens', 'sum'),
                             tokens_por_doc=('n_tokens', 'median'))

In [ ]:
# Distribución de temas por país (los 10 temas más frecuentes en total)
top_topics = noticias['topic'].value_counts().head(10).index
pd.crosstab(noticias.loc[noticias.topic.isin(top_topics), 'topic'],
            noticias.loc[noticias.topic.isin(top_topics), 'pais'], normalize='columns').round(3)

### 5.1 Ley de Heaps

El vocabulario crece de forma sublineal con el tamaño del corpus: $V(N) = k N^{\beta}$, normalmente con $0.4 < \beta < 0.7$.

In [ ]:
def curva_heaps(tokens, pasos=50):
    # Registra el tamaño del vocabulario cada cierto número de tokens
    vistos, puntos = set(), []
    paso = max(1, len(tokens) // pasos)
    for i, w in enumerate(tokens, 1):
        vistos.add(w)
        if i % paso == 0:
            puntos.append((i, len(vistos)))
    return np.array(puntos)

plt.figure(figsize=(6, 4))
for pais, grupo in noticias.groupby('pais'):
    toks = [w for t in grupo['tokens'] for w in t][:3_000_000]   # mismo presupuesto de tokens para cada país
    pts = curva_heaps(toks)
    # Ajuste log–log: la pendiente es β
    beta, logk = np.polyfit(np.log(pts[:, 0]), np.log(pts[:, 1]), 1)
    plt.plot(pts[:, 0], pts[:, 1], label=f'{pais}  β = {beta:.2f}')
plt.xlabel('tokens (N)'); plt.ylabel('vocabulario (V)'); plt.title('Ley de Heaps — noticias 2020'); plt.legend(); plt.show()

### 5.2 Vocabulario distintivo por país

*Log-odds ratio* con prior de Dirichlet informativo (Monroe, Colaresi & Quinn, 2008), como se presenta en Jurafsky & Martin.
Puntajes z positivos: palabras sobrerrepresentadas en ese país frente a los otros dos.

In [ ]:
# Log-odds con prior de Dirichlet: diferencia de log-odds dividida por su desviación estándar (puntaje z)
def log_odds_z(c_a: Counter, c_b: Counter, prior: Counter, a0: float = 0.01):
    n_a, n_b = sum(c_a.values()), sum(c_b.values())
    alpha = {w: a0 * f for w, f in prior.items()}
    a_sum = sum(alpha.values())
    out = {}
    for w, aw in alpha.items():
        ya, yb = c_a.get(w, 0), c_b.get(w, 0)
        d = np.log((ya + aw) / (n_a + a_sum - ya - aw)) - np.log((yb + aw) / (n_b + a_sum - yb - aw))
        var = 1 / (ya + aw) + 1 / (yb + aw)
        out[w] = d / np.sqrt(var)
    return pd.Series(out)

# Conteo de palabras de contenido por país; el prior es el conteo conjunto de los tres
conteos = {p: Counter(w for t in g['tokens'] for w in t if w not in STOP_ES and len(w) > 2)
           for p, g in noticias.groupby('pais')}
prior = sum(conteos.values(), Counter())

# Para cada país: sus 15 palabras con mayor z frente a los otros dos
distintivas = {}
for pais in conteos:
    resto = sum((c for p, c in conteos.items() if p != pais), Counter())
    distintivas[pais] = log_odds_z(conteos[pais], resto, prior).nlargest(15).index.tolist()
pd.DataFrame(distintivas)

---

## 6. Caso de estudio 3 — Léxico afectivo en discursos presidenciales

`lexicon_afectivo_categorias` asigna palabras en español a categorías emocionales de SEL y NRC.
Para cada discurso calculamos la proporción de tokens en cada categoría NRC.

In [ ]:
# Léxico afectivo: palabra y sus categorías emocionales
lex = cargar_df('lexicon_afectivo_categorias')
print(lex.shape)
print(lex['category_Ncr'].value_counts().head(12))
lex.head()

In [ ]:
# Diccionario palabra → conjunto de categorías NRC
lex_ncr = (lex.dropna(subset=['category_Ncr'])
              .assign(Palabra=lambda d: d['Palabra'].str.lower().str.strip())
              .drop_duplicates(['Palabra', 'category_Ncr']))
palabra2cats = lex_ncr.groupby('Palabra')['category_Ncr'].apply(set).to_dict()

def perfil_afectivo(tokens):
    c = Counter(cat for w in tokens for cat in palabra2cats.get(w, ()))
    return {cat: 1000 * n / max(len(tokens), 1) for cat, n in c.items()}   # por cada 1 000 tokens

perfiles = pd.DataFrame(pres['tokens'].map(perfil_afectivo).tolist(), index=pres['nombre']).fillna(0).round(2)
perfiles.sort_index(axis=1)

In [ ]:
# Mapa de calor: categorías emocionales por cada 1 000 tokens
fig, ax = plt.subplots(figsize=(9, 0.35 * len(perfiles) + 1))
im = ax.imshow(perfiles.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(perfiles.shape[1]), perfiles.columns, rotation=45, ha='right')
ax.set_yticks(range(perfiles.shape[0]), perfiles.index)
fig.colorbar(im, ax=ax, label='por cada 1 000 tokens'); ax.set_title('Perfil afectivo NRC — discursos presidenciales')
plt.tight_layout(); plt.show()

> **Advertencia:** los métodos de léxico ignoran la negación, la ironía y el sentido de cada palabra (*"no hay **paz**"* sigue contando como positivo). Trata estos perfiles como descriptivos, no como clasificación de sentimiento.

---

## 7. Ejercicios

1. **Catálogo:** ¿qué tres corpus usarías para comparar publicaciones en redes sobre salud de *Colombia, México y Argentina*? Justifica con filas, fechas y columna de texto.
2. **TTR:** calcula `TTR_1000` para las noticias de cada país. ¿El orden se mantiene si cambias la ventana a 500 o a 2 000 tokens?
3. **Zipf:** ajusta la pendiente de Zipf para las noticias de cada país. ¿Cambia $\alpha$ si quitas las stopwords? ¿Por qué?
4. **Palabras distintivas:** repite la sección 5.2 por `topic` en lugar de por país. ¿Qué temas tienen el vocabulario más propio de cada país?
5. **Tarea supervisada (anticipo de la Unidad 3):** carga `es_hs_project_raw_extended_v2` con `cargar_df` y entrena una línea base TF-IDF + regresión logística para la etiqueta `HS`. Reporta el F1 macro en una partición estratificada 80/20.

---

## Referencias

- Jurafsky, D., & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed., borrador), Cap. 2.
- Monroe, B. L., Colaresi, M. P., & Quinn, K. M. (2008). Fightin' words: Lexical feature selection and evaluation for identifying the content of political conflict. *Political Analysis, 16*(4), 372–403.
- Mohammad, S. M., & Turney, P. D. (2013). Crowdsourcing a word–emotion association lexicon. *Computational Intelligence, 29*(3), 436–465.
- Moreno Sandoval, L. G. *open_corpus_co_es* (v0.1.1a4) [Software]. PyPI.